# Kết nối Google Drive và thiết lập thư mục chung

In [3]:
import os
from google.colab import drive

drive.mount('/content/drive')

# Thư mục lưu 4 file CSV, kết quả và mô hình tốt nhất trên Google Drive
DATA_DIR = '/content/drive/MyDrive/BERT_FakeNews'
BEST_MODEL_DIR = f'{DATA_DIR}/best_model'
os.makedirs(DATA_DIR, exist_ok=True)

# Kiểm tra nhanh các file dữ liệu đã có trong Google Drive chưa
required_files = ['train.csv', 'val.csv', 'test.csv', 'external_10_sentences.csv']
print(f"Đang kiểm tra thư mục: {DATA_DIR}")
for fname in required_files:
    fpath = os.path.join(DATA_DIR, fname)
    status = "✅ Đã có" if os.path.exists(fpath) else "❌ CHƯA THẤY"
    print(f" - {fname}: {status}")

Mounted at /content/drive
Đang kiểm tra thư mục: /content/drive/MyDrive/BERT_FakeNews
 - train.csv: ✅ Đã có
 - val.csv: ✅ Đã có
 - test.csv: ✅ Đã có
 - external_10_sentences.csv: ✅ Đã có


# Cài đặt thư viện, Import

In [4]:
!pip install -q transformers datasets scikit-learn

import os
import re
import shutil
import numpy as np
import pandas as pd
import torch
import torch.nn.functional as F
from datasets import Dataset
from transformers import (
    BertTokenizer,
    BertForSequenceClassification,
    Trainer,
    TrainingArguments,
    set_seed
)
from sklearn.metrics import accuracy_score, f1_score, classification_report, confusion_matrix

# Đọc dữ liệu và Ánh xạ nhãn

In [5]:
# 1. Đọc dữ liệu

train_df = pd.read_csv(f'{DATA_DIR}/train.csv')
val_df = pd.read_csv(f'{DATA_DIR}/val.csv')
test_df = pd.read_csv(f'{DATA_DIR}/test.csv')

# 2. Map nhãn chuẩn: fake=0, real=1
label_map = {'fake': 0, 'real': 1}
for name, df in [('Train', train_df), ('Val', val_df), ('Test', test_df)]:
    df['label'] = df['label'].astype(str).str.strip().str.lower().map(label_map)
    assert df['label'].isna().sum() == 0, f"Có nhãn không hợp lệ trong tập {name}!"
    print(f"{name}: {len(df)} mẫu | Phân bố nhãn (0=Fake, 1=Real): {df['label'].value_counts().to_dict()}")

# 3. Chuyển đổi sang format Dataset của Hugging Face
train_dataset = Dataset.from_pandas(train_df[['tweet', 'label']])
val_dataset = Dataset.from_pandas(val_df[['tweet', 'label']])
test_dataset = Dataset.from_pandas(test_df[['tweet', 'label']])

Train: 6420 mẫu | Phân bố nhãn (0=Fake, 1=Real): {1: 3360, 0: 3060}
Val: 2140 mẫu | Phân bố nhãn (0=Fake, 1=Real): {1: 1120, 0: 1020}
Test: 2140 mẫu | Phân bố nhãn (0=Fake, 1=Real): {1: 1120, 0: 1020}


# Tokenizer(Tiền xử lý văn bản)

In [6]:
# Tải tokenizer của BERT-base
tokenizer = BertTokenizer.from_pretrained('bert-base-uncased')

def tokenize_function(examples):
    # Padding và cắt gọn văn bản
    return tokenizer(examples['tweet'], padding='max_length', truncation=True, max_length=128)

# Áp dụng tokenizer lên toàn bộ tập dữ liệu
train_tokenized = train_dataset.map(tokenize_function, batched=True)
val_tokenized = val_dataset.map(tokenize_function, batched=True)
test_tokenized = test_dataset.map(tokenize_function, batched=True)

tokenizer_config.json:   0%|          | 0.00/48.0 [00:00<?, ?B/s]

vocab.txt:   0%|          | 0.00/232k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/466k [00:00<?, ?B/s]

Map:   0%|          | 0/6420 [00:00<?, ? examples/s]

Map:   0%|          | 0/2140 [00:00<?, ? examples/s]

Map:   0%|          | 0/2140 [00:00<?, ? examples/s]

# Huấn luyện 3 seed và Lưu mô hình tốt nhất trên Drive

In [7]:
def compute_metrics(eval_pred):
    logits, labels = eval_pred
    predictions = np.argmax(logits, axis=-1)
    return {
        'accuracy': accuracy_score(labels, predictions),
        'f1_weighted': f1_score(labels, predictions, average='weighted')
    }

seeds = [42, 123, 2024]
id2label = {0: "Fake", 1: "Real"}
label2id = {"Fake": 0, "Real": 1}

val_summary = []

for seed in seeds:
    print(f"\n{'='*45}")
    print(f"BẮT ĐẦU HUẤN LUYỆN VỚI SEED: {seed}")
    print(f"{'='*45}")

    set_seed(seed)

    model = BertForSequenceClassification.from_pretrained(
        'bert-base-uncased',
        num_labels=2,
        id2label=id2label,
        label2id=label2id
    )

    output_dir = f'./bert_base_seed_{seed}'

    training_args = TrainingArguments(
        output_dir=output_dir,
        num_train_epochs=3,
        per_device_train_batch_size=16,
        per_device_eval_batch_size=16,
        learning_rate=2e-5,
        weight_decay=0.01,
        eval_strategy="epoch",
        save_strategy="epoch",
        save_total_limit=1,
        metric_for_best_model="f1_weighted",
        load_best_model_at_end=True,
        seed=seed,
        report_to="none"
    )

    trainer = Trainer(
        model=model,
        args=training_args,
        train_dataset=train_tokenized,
        eval_dataset=val_tokenized,
        compute_metrics=compute_metrics
    )

    trainer.train()

    val_results = trainer.evaluate()
    print(f"\nKết quả trên tập Validation (Seed {seed}):")
    print(val_results)

    # Lưu mô hình tốt nhất của từng seed vào đường dẫn cố định
    trainer.save_model(f'{output_dir}/best_model')
    tokenizer.save_pretrained(f'{output_dir}/best_model')

    val_summary.append({
        'Seed': seed,
        'Val Loss': val_results['eval_loss'],
        'Accuracy': val_results['eval_accuracy'],
        'F1 Weighted': val_results['eval_f1_weighted']
    })

df_val = pd.DataFrame(val_summary)
best_seed = int(df_val.loc[df_val['F1 Weighted'].idxmax(), 'Seed'])

print("\nBẢNG TỔNG HỢP KẾT QUẢ VALIDATION:")
print(df_val)
print(f"=> Accuracy Trung bình (Val):    {df_val['Accuracy'].mean():.6f} ± {df_val['Accuracy'].std():.6f}")
print(f"=> F1 Weighted Trung bình (Val): {df_val['F1 Weighted'].mean():.6f} ± {df_val['F1 Weighted'].std():.6f}")
print(f"=> Seed tốt nhất trên tập Validation: {best_seed}")

# Copy mô hình của seed tốt nhất sang Google Drive
if os.path.exists(BEST_MODEL_DIR):
    shutil.rmtree(BEST_MODEL_DIR)
shutil.copytree(f'./bert_base_seed_{best_seed}/best_model', BEST_MODEL_DIR)
print(f"Đã lưu mô hình tốt nhất (Seed {best_seed}) vào Google Drive: {BEST_MODEL_DIR}")


BẮT ĐẦU HUẤN LUYỆN VỚI SEED: 42


config.json:   0%|          | 0.00/570 [00:00<?, ?B/s]

model.safetensors: reconstructing file:   0%|          |  0.00B /  440MB            

model.safetensors: downloading bytes:           |  0.00B            

Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

[transformers] BertForSequenceClassification LOAD REPORT from: bert-base-uncased
Key                                        | Status     | 
-------------------------------------------+------------+-
cls.predictions.bias                       | UNEXPECTED | 
cls.seq_relationship.bias                  | UNEXPECTED | 
cls.predictions.transform.dense.weight     | UNEXPECTED | 
cls.predictions.transform.LayerNorm.bias   | UNEXPECTED | 
cls.predictions.transform.LayerNorm.weight | UNEXPECTED | 
cls.seq_relationship.weight                | UNEXPECTED | 
cls.predictions.transform.dense.bias       | UNEXPECTED | 
classifier.weight                          | MISSING    | 
classifier.bias                            | MISSING    | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING:	those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


... (27 outputs left out)

Training Loss,Validation Loss,Epoch,Accuracy,F1 Weighted
0.048463,0.130571,3,0.971963,0.971954



Kết quả trên tập Validation (Seed 2024):
{'eval_loss': 0.13057133555412292, 'eval_accuracy': 0.9719626168224299, 'eval_f1_weighted': 0.9719543659398062}


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]


BẢNG TỔNG HỢP KẾT QUẢ VALIDATION:
   Seed  Val Loss  Accuracy  F1 Weighted
0    42  0.137352  0.971963     0.971946
1   123  0.133601  0.974299     0.974280
2  2024  0.130571  0.971963     0.971954
=> Accuracy Trung bình (Val):    0.972741 ± 0.001349
=> F1 Weighted Trung bình (Val): 0.972727 ± 0.001345
=> Seed tốt nhất trên tập Validation: 123
Đã lưu mô hình tốt nhất (Seed 123) vào Google Drive: /content/drive/MyDrive/BERT_FakeNews/best_model


# ĐÁNH GIÁ TRÊN TẬP TEST CHO 3 SEED

In [8]:
print("="*50)
print("TIẾN HÀNH ĐÁNH GIÁ TRÊN TẬP TEST (test.csv)")
print("="*50)

eval_args = TrainingArguments(
    output_dir='./tmp_eval',
    per_device_eval_batch_size=16,
    report_to='none'
)

test_results = []

for seed in seeds:
    model_path = f'./bert_base_seed_{seed}/best_model'
    print(f"\nĐang nạp mô hình (Seed {seed}) từ: {model_path}")

    model_test = BertForSequenceClassification.from_pretrained(model_path)
    model_test.to('cuda' if torch.cuda.is_available() else 'cpu')

    trainer_test = Trainer(
        model=model_test,
        args=eval_args,
        compute_metrics=compute_metrics
    )

    pred_output = trainer_test.predict(test_tokenized)
    preds = np.argmax(pred_output.predictions, axis=1)
    labels = pred_output.label_ids
    metrics = pred_output.metrics

    acc = metrics['test_accuracy']
    f1 = metrics['test_f1_weighted']

    print(f"--> Kết quả Test (Seed {seed}): Accuracy: {acc:.4f} | F1: {f1:.4f}")
    print("\nBáo cáo chi tiết từng lớp (Classification Report):")
    print(classification_report(labels, preds, target_names=["Fake (0)", "Real (1)"], digits=4))
    print("Ma trận nhầm lẫn (Confusion Matrix):")
    print(confusion_matrix(labels, preds))
    print("-" * 50)

    test_results.append({'Seed': seed, 'Accuracy': acc, 'F1 Weighted': f1})

df_test = pd.DataFrame(test_results)
print("\nBẢNG TỔNG HỢP KẾT QUẢ TEST:")
print(df_test)
print(f"\n=> Accuracy Trung bình (Test):    {df_test['Accuracy'].mean():.6f} ± {df_test['Accuracy'].std():.6f}")
print(f"=> F1 Weighted Trung bình (Test): {df_test['F1 Weighted'].mean():.6f} ± {df_test['F1 Weighted'].std():.6f}")

TIẾN HÀNH ĐÁNH GIÁ TRÊN TẬP TEST (test.csv)

Đang nạp mô hình (Seed 42) từ: ./bert_base_seed_42/best_model


Loading weights:   0%|          | 0/201 [00:00<?, ?it/s]

--> Kết quả Test (Seed 42): Accuracy: 0.9738 | F1: 0.9738

Báo cáo chi tiết từng lớp (Classification Report):
              precision    recall  f1-score   support

    Fake (0)     0.9782    0.9667    0.9724      1020
    Real (1)     0.9700    0.9804    0.9751      1120

    accuracy                         0.9738      2140
   macro avg     0.9741    0.9735    0.9738      2140
weighted avg     0.9739    0.9738    0.9738      2140

Ma trận nhầm lẫn (Confusion Matrix):
[[ 986   34]
 [  22 1098]]
--------------------------------------------------

Đang nạp mô hình (Seed 123) từ: ./bert_base_seed_123/best_model


Loading weights:   0%|          | 0/201 [00:00<?, ?it/s]

--> Kết quả Test (Seed 123): Accuracy: 0.9734 | F1: 0.9733

Báo cáo chi tiết từng lớp (Classification Report):
              precision    recall  f1-score   support

    Fake (0)     0.9820    0.9618    0.9718      1020
    Real (1)     0.9658    0.9839    0.9748      1120

    accuracy                         0.9734      2140
   macro avg     0.9739    0.9728    0.9733      2140
weighted avg     0.9735    0.9734    0.9733      2140

Ma trận nhầm lẫn (Confusion Matrix):
[[ 981   39]
 [  18 1102]]
--------------------------------------------------

Đang nạp mô hình (Seed 2024) từ: ./bert_base_seed_2024/best_model


Loading weights:   0%|          | 0/201 [00:00<?, ?it/s]

--> Kết quả Test (Seed 2024): Accuracy: 0.9762 | F1: 0.9762

Báo cáo chi tiết từng lớp (Classification Report):
              precision    recall  f1-score   support

    Fake (0)     0.9831    0.9667    0.9748      1020
    Real (1)     0.9701    0.9848    0.9774      1120

    accuracy                         0.9762      2140
   macro avg     0.9766    0.9757    0.9761      2140
weighted avg     0.9763    0.9762    0.9762      2140

Ma trận nhầm lẫn (Confusion Matrix):
[[ 986   34]
 [  17 1103]]
--------------------------------------------------

BẢNG TỔNG HỢP KẾT QUẢ TEST:
   Seed  Accuracy  F1 Weighted
0    42  0.973832     0.973824
1   123  0.973364     0.973350
2  2024  0.976168     0.976158

=> Accuracy Trung bình (Test):    0.974455 ± 0.001502
=> F1 Weighted Trung bình (Test): 0.974444 ± 0.001503


# Error Analysis (Phân tích lỗi)

In [9]:
print(f"Đang nạp mô hình tốt nhất từ: {BEST_MODEL_DIR}")
best_model = BertForSequenceClassification.from_pretrained(BEST_MODEL_DIR)
best_model.to('cuda' if torch.cuda.is_available() else 'cpu')

eval_args = TrainingArguments(
    output_dir='./tmp_eval',
    per_device_eval_batch_size=16,
    report_to='none'
)

trainer_val = Trainer(
    model=best_model,
    args=eval_args,
    compute_metrics=compute_metrics
)

pred_output = trainer_val.predict(val_tokenized)
probs = F.softmax(torch.tensor(pred_output.predictions), dim=-1).numpy()
preds = np.argmax(pred_output.predictions, axis=1)
labels = pred_output.label_ids

error_indices = np.where(preds != labels)[0]
fn_count = int(np.sum((labels == 1) & (preds == 0)))
fp_count = int(np.sum((labels == 0) & (preds == 1)))

print(f"\nTổng số câu dự đoán sai trên Validation: {len(error_indices)} / {len(labels)}")
print(f"- Real (1) bị đoán nhầm thành Fake (0) [False Negative]: {fn_count}")
print(f"- Fake (0) bị đoán nhầm thành Real (1) [False Positive]: {fp_count}\n")

val_error_rows = []
for idx, i in enumerate(error_indices):
    tweet = val_dataset[int(i)]['tweet']
    true_label = int(val_dataset[int(i)]['label'])
    pred_label = int(preds[i])
    conf = float(probs[i][pred_label]) * 100

    true_text = "Real (1)" if true_label == 1 else "Fake (0)"
    pred_text = "Real (1)" if pred_label == 1 else "Fake (0)"

    val_error_rows.append({
        'index': int(i),
        'true_label': true_text,
        'pred_label': pred_text,
        'confidence_%': round(conf, 2),
        'tweet': tweet
    })

    if idx < 10:
        print(f"--- Mẫu sai số {idx + 1} (Index: {i}) ---")
        print(f"Thực tế: {true_text} | Mô hình đoán: {pred_text} (Độ tin cậy: {conf:.2f}%)")
        print(f"Nội dung: {tweet}\n")

pd.DataFrame(val_error_rows).to_csv(f'{DATA_DIR}/val_errors.csv', index=False)
print(f"Đã lưu toàn bộ các mẫu sai trên tập Validation tại: {DATA_DIR}/val_errors.csv")

Đang nạp mô hình tốt nhất từ: /content/drive/MyDrive/BERT_FakeNews/best_model


Loading weights:   0%|          | 0/201 [00:00<?, ?it/s]


Tổng số câu dự đoán sai trên Validation: 55 / 2140
- Real (1) bị đoán nhầm thành Fake (0) [False Negative]: 14
- Fake (0) bị đoán nhầm thành Real (1) [False Positive]: 41

--- Mẫu sai số 1 (Index: 13) ---
Thực tế: Real (1) | Mô hình đoán: Fake (0) (Độ tin cậy: 99.85%)
Nội dung: Tomorrow April 6 we will pass 10000 coronavirus deaths. We passed 5000 on April 2. We passed 1000 on March 26. We passed 100 on March 18.

--- Mẫu sai số 2 (Index: 49) ---
Thực tế: Fake (0) | Mô hình đoán: Real (1) (Độ tin cậy: 91.74%)
Nội dung: A coronavirus vaccine: Where does it stand? Experts we checked with remain optimistic about a vaccine in the first quarter of 2021. https://t.co/XAwbq9IqrH https://t.co/ONglQMMwFc

--- Mẫu sai số 3 (Index: 61) ---
Thực tế: Fake (0) | Mô hình đoán: Real (1) (Độ tin cậy: 99.92%)
Nội dung: The Global #dietarysupplements market is anticipated to grow at a #CAGR of around 7.50% during 2020-25. 
Check the full report:- https://t.co/71J6LqCvgr

#food #beverage #marketresearch 

# Thử nghiệm và phân tích lỗi trên 10 câu thực tế

In [10]:
print("\n" + "="*60)
print("THỬ NGHIỆM & PHÂN TÍCH LỖI TRÊN 10 CÂU NGOÀI (external_10_sentences.csv)")
print("="*60)

print(f"Đang nạp mô hình tốt nhất từ: {BEST_MODEL_DIR}")
tokenizer = BertTokenizer.from_pretrained(BEST_MODEL_DIR)
best_model = BertForSequenceClassification.from_pretrained(BEST_MODEL_DIR)
best_model.to('cuda' if torch.cuda.is_available() else 'cpu')
best_model.eval()

ext_df = pd.read_csv(f'{DATA_DIR}/external_10_sentences.csv')
label_map = {'fake': 0, 'real': 1}
ext_df['true_label'] = ext_df['label'].astype(str).str.strip().str.lower().map(label_map)
assert ext_df['true_label'].isna().sum() == 0, "Có nhãn không hợp lệ trong file external_10_sentences.csv!"

def diagnose_error(text, true_lbl, pred_lbl):
    reasons = []
    has_url = bool(re.search(r'http[s]?://|t\.co', text))
    has_hashtag = '#' in text
    has_mention = '@' in text
    has_numbers = bool(re.search(r'\d+', text))
    has_negation = bool(re.search(r"\b(no|not|never|does not|do not|cannot|can't|don't|doesn't)\b", text.lower()))

    if true_lbl == 1 and pred_lbl == 0:
        reasons.append("False Negative (Tin thật bị đoán nhầm thành Tin giả).")
        if not (has_url or has_hashtag or has_mention):
            reasons.append("Câu văn chuẩn, không chứa đặc trưng định dạng Twitter trong tập Train (link t.co, #hashtag, @mention) nên dễ bị ảnh hưởng bởi lệch miền dữ liệu (domain shift).")
        if not has_numbers:
            reasons.append("Không có số liệu thống kê (số ca nhiễm/tử vong), vốn là tín hiệu mạnh của nhãn Real trong tập Constraint@AAAI2021.")
        if has_negation:
            reasons.append("Có cấu trúc phủ định khiến mô hình dễ nhầm lẫn ngữ nghĩa.")
    elif true_lbl == 0 and pred_lbl == 1:
        reasons.append("False Positive (Tin giả bị đoán nhầm thành Tin thật).")
        if has_numbers:
            reasons.append("Câu tin giả có chèn con số cụ thể khiến mô hình nhầm với phong cách bản tin thống kê.")
        reasons.append("Cách diễn đạt mang cấu trúc tường thuật/khẳng định tương tự tin tức chính thống, đánh lừa bộ phân loại dựa trên bề mặt ngôn ngữ.")

    return " ".join(reasons)

y_true, y_pred = [], []
results_rows, wrong_preds = [], []

for _, row in ext_df.iterrows():
    sample_id = int(row['id'])
    text = str(row['sentence'])
    true_lbl = int(row['true_label'])

    inputs = tokenizer(text, padding='max_length', truncation=True, max_length=128, return_tensors="pt")
    inputs = {k: v.to(best_model.device) for k, v in inputs.items()}

    with torch.no_grad():
        outputs = best_model(**inputs)
        probs = F.softmax(outputs.logits, dim=-1)[0].cpu().numpy()
        pred_lbl = int(probs.argmax())
        confidence = float(probs[pred_lbl]) * 100

    y_true.append(true_lbl)
    y_pred.append(pred_lbl)

    true_name = "Real (1)" if true_lbl == 1 else "Fake (0)"
    pred_name = "Real (1)" if pred_lbl == 1 else "Fake (0)"
    status = "✅ ĐÚNG" if true_lbl == pred_lbl else "❌ SAI"

    print(f"Câu {sample_id:02d} [{status}] | Thực tế: {true_name} | Dự đoán: {pred_name} (Độ tin cậy: {confidence:.2f}%)")
    print(f"   Nội dung: {text}\n")

    results_rows.append({
        'id': sample_id,
        'sentence': text,
        'true_label': true_name,
        'pred_label': pred_name,
        'confidence_%': round(confidence, 2),
        'correct': true_lbl == pred_lbl
    })

    if true_lbl != pred_lbl:
        wrong_preds.append({
            'id': sample_id,
            'text': text,
            'true_lbl': true_lbl,
            'pred_lbl': pred_lbl,
            'true_name': true_name,
            'pred_name': pred_name,
            'confidence': confidence
        })

acc = accuracy_score(y_true, y_pred)
f1 = f1_score(y_true, y_pred, average='weighted')

print("-" * 60)
print(f"KẾT QUẢ TỔNG HỢP TRÊN 10 CÂU NGOÀI: Accuracy = {acc:.4f} ({sum(r['correct'] for r in results_rows)}/10) | F1 Weighted = {f1:.4f}")
print("\nBáo cáo chi tiết (Classification Report):")
print(classification_report(y_true, y_pred, target_names=["Fake (0)", "Real (1)"], digits=4, zero_division=0))
print("Ma trận nhầm lẫn (Confusion Matrix):")
print(confusion_matrix(y_true, y_pred))

print("\n" + "-" * 60)
print("PHÂN TÍCH CHI TIẾT CÁC CÂU DỰ ĐOÁN SAI:")
if len(wrong_preds) == 0:
    print("Mô hình dự đoán đúng 10/10 câu thực tế ngoài dữ liệu.")
else:
    for item in wrong_preds:
        print(f"-> Câu {item['id']:02d}: {item['text']}")
        print(f"   Nhầm lẫn: Thực tế là {item['true_name']} nhưng mô hình đoán {item['pred_name']} ({item['confidence']:.2f}%).")
        print(f"   Nguyên nhân dự kiến: {diagnose_error(item['text'], item['true_lbl'], item['pred_lbl'])}\n")

pd.DataFrame(results_rows).to_csv(f'{DATA_DIR}/external_10_results.csv', index=False)
print(f"Đã lưu bảng kết quả dự đoán tại: {DATA_DIR}/external_10_results.csv")


THỬ NGHIỆM & PHÂN TÍCH LỖI TRÊN 10 CÂU NGOÀI (external_10_sentences.csv)
Đang nạp mô hình tốt nhất từ: /content/drive/MyDrive/BERT_FakeNews/best_model


Loading weights:   0%|          | 0/201 [00:00<?, ?it/s]

Câu 01 [❌ SAI] | Thực tế: Real (1) | Dự đoán: Fake (0) (Độ tin cậy: 99.63%)
   Nội dung: WHO recommends washing your hands frequently with soap and water to reduce the risk of infection.

Câu 02 [❌ SAI] | Thực tế: Real (1) | Dự đoán: Fake (0) (Độ tin cậy: 99.97%)
   Nội dung: Wearing a mask in crowded indoor places can help reduce the spread of COVID-19.

Câu 03 [❌ SAI] | Thực tế: Real (1) | Dự đoán: Fake (0) (Độ tin cậy: 99.87%)
   Nội dung: The health ministry reported 1,250 new confirmed COVID-19 cases and 14 deaths in the last 24 hours.

Câu 04 [❌ SAI] | Thực tế: Real (1) | Dự đoán: Fake (0) (Độ tin cậy: 99.89%)
   Nội dung: Vaccines are approved only after clinical trials show they are safe and effective.

Câu 05 [❌ SAI] | Thực tế: Real (1) | Dự đoán: Fake (0) (Độ tin cậy: 98.74%)
   Nội dung: People with symptoms such as fever, cough or loss of taste or smell should get tested and isolate.

Câu 06 [✅ ĐÚNG] | Thực tế: Fake (0) | Dự đoán: Fake (0) (Độ tin cậy: 99.98%)
   Nội dung: 

In [11]:
tokenizer = BertTokenizer.from_pretrained(BEST_MODEL_DIR)
best_model = BertForSequenceClassification.from_pretrained(BEST_MODEL_DIR)
best_model.to('cuda' if torch.cuda.is_available() else 'cpu')
best_model.eval()

def predict_fake_news(text):
    print(f"Đang phân tích câu: '{text}'")

    inputs = tokenizer(text, padding='max_length', truncation=True, max_length=128, return_tensors="pt")
    inputs = {k: v.to(best_model.device) for k, v in inputs.items()}

    with torch.no_grad():
        outputs = best_model(**inputs)
        probs = F.softmax(outputs.logits, dim=-1)[0].cpu().numpy()

    prediction = int(probs.argmax())
    confidence = float(probs[prediction]) * 100

    label_name = f"{best_model.config.id2label[prediction]} ({'Tin thật' if prediction == 1 else 'Tin giả'})"
    print(f"=> Kết quả dự đoán: {label_name} | Độ tin cậy: {confidence:.2f}%\n")

cau_thu_nghiem = "States reported 1121 deaths a small rise from last Tuesday. Southern states reported 640 of those deaths. https://t.co/YASGRTT4ux"
predict_fake_news(cau_thu_nghiem)

Loading weights:   0%|          | 0/201 [00:00<?, ?it/s]

Đang phân tích câu: 'States reported 1121 deaths a small rise from last Tuesday. Southern states reported 640 of those deaths. https://t.co/YASGRTT4ux'
=> Kết quả dự đoán: Real (Tin thật) | Độ tin cậy: 99.94%



In [12]:
!zip -r "{DATA_DIR}/config_seed_{best_seed}.zip" ./bert_base_seed_{best_seed} -x "*model.safetensors" "*optimizer.pt" "*scheduler.pt" "*rng_state.pth"

updating: bert_base_seed_123/ (stored 0%)
updating: bert_base_seed_123/best_model/ (stored 0%)
updating: bert_base_seed_123/best_model/tokenizer.json (deflated 71%)
updating: bert_base_seed_123/best_model/tokenizer_config.json (deflated 43%)
updating: bert_base_seed_123/best_model/config.json (deflated 53%)
updating: bert_base_seed_123/best_model/training_args.bin (deflated 53%)
updating: bert_base_seed_123/checkpoint-1206/ (stored 0%)
updating: bert_base_seed_123/checkpoint-1206/config.json (deflated 53%)
updating: bert_base_seed_123/checkpoint-1206/trainer_state.json (deflated 64%)
updating: bert_base_seed_123/checkpoint-1206/training_args.bin (deflated 53%)


In [13]:
from google.colab import files
import os
import shutil

DATA_DIR = '/content/drive/MyDrive/BERT_FakeNews'
ZIP_OUTPUT_W_EXT = '/content/Ly_Bert_Full'
ZIP_PATH = ZIP_OUTPUT_W_EXT + '.zip'

# Kiểm tra thư mục nguồn có tồn tại hay không trước khi nén
if os.path.exists(DATA_DIR):
    print(f"Đang nén thư mục {DATA_DIR}...")
    # Sử dụng shutil.make_archive để nén thư mục an toàn bằng Python
    shutil.make_archive(ZIP_OUTPUT_W_EXT, 'zip', root_dir=os.path.dirname(DATA_DIR), base_dir=os.path.basename(DATA_DIR))

    if os.path.exists(ZIP_PATH):
        print("Dung lượng file zip:", round(os.path.getsize(ZIP_PATH) / (1024**2), 2), "MB")
        # Tải trực tiếp về máy
        files.download(ZIP_PATH)
    else:
        print("Lỗi: Không thể tạo file zip.")
else:
    print(f"Lỗi: Thư mục nguồn không tồn tại tại {DATA_DIR}. Vui lòng kiểm tra lại kết nối Google Drive!")

Đang nén thư mục /content/drive/MyDrive/BERT_FakeNews...
Dung lượng file zip: 388.0 MB


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>